In [18]:
#Assorted imports
import numpy as np
import pandas as pd
import h5py
import vaex
import pynbody
from pynbody.array import SimArray

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import PathPatch
from matplotlib.path import Path

from astropy import units as u
from astropy.io import ascii, fits
from astropy.table import Table, vstack
from astropy.coordinates import SkyCoord,CartesianRepresentation,match_coordinates_sky

import functions as fn 
import matched_filter_ani as mf
import os

np.random.seed(0)

In [19]:
s = pynbody.load('/home/christenc/Storage/Cosmo/MM/r615.romulus25.3072g1HsbBH/r615.romulus25.3072g1HsbBH.004096/r615.romulus25.3072g1HsbBH.004096')

h = s.halos(halo_numbers='v1')  # Load the halos using the original AHF numbering system
h.load_all()
unique_halo_ids = list(h.keys())


In [23]:
num = 615
main_halo = h[1] ### IF WORKING WITH MARVEL, CHANGE TO h[num].


In [24]:
pynbody.analysis.halo.center
cen = main_halo.mean_by_mass('pos')

sp = s[pynbody.filt.Sphere(SimArray([200], "kpc"), cen)].load_copy()

s.physical_units()

partids_snap = sp.s['iord']

with h5py.File('/home/christenc/Storage/Cosmo/MM/r615.romulus25.3072g1HsbBH/r615_allhalostardata_consolidated2.h5','r') as f:
    hostids = f['host_IDs'].asstr()[:] 
    partids_h5 = f['particle_IDs'][:]



In [26]:
box = 'r' 
D = 2000 
mlim_str = '26p5' 
name = f'{box}_4096_{num}_data_{D}_{mlim_str}'

#dwarfcatpath = f"/home/otteleno/MAP/raw_data/{box}_{num}/survey.{name}.0.h5" ###MARVEL CONVENTION
dwarfcatpath = f"/home/otteleno/MAP/raw_data/{box}_{num}/survey.MM_{box}{num}_data_2000_26p5.0.h5" ###MM CONVENTION
vdwarfcat = vaex.open(dwarfcatpath)
dwarfcat = pd.DataFrame(vdwarfcat,columns=vdwarfcat.column_names)

size_kpc = 40 
pdist = (size_kpc/D) * (180/np.pi) 
year = 10
mlim = '25'
c1='px'
c2='py'
edgelength = 10 
plotdir = f'{box}_4096_{num}' 

In [27]:
dwarfcat

,age,dec,dmod,feh,glat,glon,grav,lsst_g,lsst_g_Err,lsst_g_Intrinsic,...,py,pz,ra,rad,smass,teff,vr,vx,vy,vz
0,10.144734,-27.095028,26.504463,-4.054715,-89.966200,112.340901,0.732200,24.524052,-0.0,-1.980411,...,1.090933,-1999.367242,12.852500,1999.367590,0.787524,4511.346680,101.828519,-116.877119,-119.486383,-101.867524
1,10.144734,-27.157780,26.505907,-4.024374,-89.970318,-62.895834,1.805265,26.259260,-0.0,-0.246647,...,-0.922620,-2000.697127,12.862865,2000.697396,0.787121,4962.934570,81.894456,-155.750067,-223.559564,-81.828134
2,10.144734,-27.132148,26.505358,-4.083127,-89.978636,-136.555775,2.087764,26.807741,-0.0,0.302382,...,-0.512869,-2000.191555,12.883081,2000.191694,0.786821,5070.593750,111.723604,-162.864722,-230.625347,-111.620385
3,10.144708,-27.095909,26.506067,-3.971524,-89.911220,-168.411992,2.116079,26.863970,-0.0,0.357902,...,-0.622767,-2000.842354,12.952363,2000.844756,0.786786,5081.625977,123.124908,-184.969199,-274.639251,-122.758806
4,10.144681,-27.204447,26.505749,-4.025951,-89.921522,-70.919181,2.047623,26.728626,0.0,0.222878,...,-2.589598,-2000.549480,12.880602,2000.551357,0.786868,5055.708984,115.358625,-162.785941,-265.347814,-115.088144
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9943278,4.770065,-27.131688,26.508453,-0.498457,-89.967734,-140.948119,4.394263,25.525530,-0.0,-0.982924,...,-0.710678,-2003.044463,12.895527,2003.044780,5.869479,20776.623047,186.397812,-168.370468,-267.551739,-186.229281
9943279,4.770065,-27.131433,26.508498,-0.498457,-89.968460,-141.272647,4.416214,25.839500,0.0,-0.668997,...,-0.689826,-2003.085777,12.894736,2003.086081,5.172175,19486.779297,186.374254,-167.963962,-267.103534,-186.210167
9943280,4.770065,-27.131468,26.508492,-0.498457,-89.968247,-141.248256,4.303440,24.155256,0.0,-2.353237,...,-0.694859,-2003.080303,12.894973,2003.080611,10.147604,27165.408203,186.893959,-168.649063,-266.883955,-186.728518
9943281,4.770065,-27.131162,26.508457,-0.498457,-89.968927,-141.687735,4.375137,25.217405,0.0,-1.291051,...,-0.673461,-2003.047690,12.894239,2003.047984,6.642207,22124.328125,186.286443,-168.806057,-267.164264,-186.124812


In [28]:
ananke_data = np.column_stack([dwarfcat['px'],dwarfcat['py'], dwarfcat['pz']])
ananke_masses = dwarfcat['smass']
ananke_total_mass = np.sum(ananke_masses)

cm_current_ananke = [0,0,0]
for i in range(len(ananke_data)):
    cm_current_ananke = cm_current_ananke + ananke_data[i] * ananke_masses[i]
cm_ananke = [x/ananke_total_mass for x in cm_current_ananke]


ananke_data_centered = ananke_data - cm_ananke
print(ananke_data_centered)

[[-0.27780078  0.93430432  0.60149267]
 [ 0.64274629 -1.07924793 -0.72839239]
 [-0.37097213 -0.66949705 -0.22281954]
 ...
 [-0.69518883 -0.85148728 -3.11156805]
 [-0.68184126 -0.83008966 -3.07895469]
 [-0.69113022 -0.9081122  -3.09173736]]


In [29]:
def fibonacci_angler(n):

    golden_ratio = (1 + np.sqrt(5))/2

    i_array = np.linspace(0,n-1,n)
    z_array = 1 - i_array/((n-1)) ###ONLY GOES HALF WAY DOWN
    radius_array = np.sqrt(1-z_array**2)

    declination_array = np.pi/2-np.arcsin(z_array)
    azimuthal_array = 2*np.pi * i_array/golden_ratio
    

    return declination_array, azimuthal_array

In [30]:
declination_array, azimuthal_array = fibonacci_angler(24)

In [31]:
def new_axes(host_ids):

    #Calculates statistics for the entire galaxy
    
    m = sp.s['mass']
    x = sp.s['pos'][:, 0]
    y = sp.s['pos'][:, 1]
    z = sp.s['pos'][:, 2]
    vx = sp.s['vel'][:, 0]
    vy = sp.s['vel'][:, 1]
    vz = sp.s['vel'][:, 2]

    particle_array_all = np.column_stack((m,x,y,z,vx,vy,vz))
    pos_array_all = particle_array_all[:, 1:4]
    vel_array_all = particle_array_all[:, 4:7]
    
    total_mass_all=np.sum(m)
    
    cm_current_all = [0,0,0]
    for i in range(len(particle_array_all)):
        cm_current_all = cm_current_all + particle_array_all[i][0] * pos_array_all[i]
    cm_all = cm_current_all/total_mass_all
    
    avg_vel_current_all = [0,0,0]
    for i in range(len(particle_array_all)):
        avg_vel_current_all = avg_vel_current_all + particle_array_all[i][0] * vel_array_all[i]
    avg_vel_all = avg_vel_current_all/total_mass_all
    
    pos_array_centered_all = pos_array_all - cm_all
    vel_array_centered_all = vel_array_all - avg_vel_all


    #Calculates statistics only for the halo of interest, but adjusts them using the entire galaxy
    
    _, idloc_snap, idloc_h5 = np.intersect1d(partids_snap, partids_h5, return_indices = True) 
                                                                                             
    
    progenitor = hostids[idloc_h5] 
    mask = np.isin(progenitor, host_ids) 
    
    m_halo = m[idloc_snap][mask] 
    x_halo = x[idloc_snap][mask]
    y_halo = y[idloc_snap][mask]
    z_halo = z[idloc_snap][mask]
    vx_halo = vx[idloc_snap][mask]
    vy_halo = vy[idloc_snap][mask]
    vz_halo = vz[idloc_snap][mask]

    particle_array_halo = np.column_stack((m_halo,x_halo,y_halo,z_halo,vx_halo,vy_halo,vz_halo)) 
    pos_array_halo = particle_array_halo[:, 1:4]
    vel_array_halo = particle_array_halo[:, 4:7]
    pos_array_centered_halo = pos_array_halo - cm_all 
    vel_array_centered_halo = vel_array_halo - avg_vel_all 
    
    total_mass_halo=np.sum(m_halo)

    cm_current_halo = [0,0,0]
    for i in range(len(particle_array_halo)):
        cm_current_halo = cm_current_halo + particle_array_halo[i][0] * pos_array_centered_halo[i]
    cm_halo = [x/total_mass_halo for x in cm_current_halo]
    
    ang_mom_halo = [0,0,0]
    for i in range(len(particle_array_halo)):
        ang_mom_halo = ang_mom_halo + particle_array_halo[i][0] * np.cross(pos_array_centered_halo[i], vel_array_centered_halo[i])

    #Creates new basis and transforms data

    z_prime = ang_mom_halo/np.linalg.norm(ang_mom_halo)
    y_prime_unnormed = np.cross(ang_mom_halo, cm_halo)
    y_prime =  y_prime_unnormed/np.linalg.norm(y_prime_unnormed)
    x_prime = np.cross(z_prime, y_prime)
    transform_matrix = np.row_stack((x_prime, y_prime, z_prime)) 
    
    return transform_matrix
    
    

In [32]:
transform_matrix = new_axes('4032_2')

In [33]:
print(transform_matrix)

[[ 0.90620936  0.38696277 -0.17042424]
 [ 0.32454004 -0.37819696  0.86697222]
 [-0.27103204  0.84096782  0.46831053]]


In [34]:
def angle_view(coordinate_transform, declination, azimuthal,save = False, plot=True):

    '''Calculates 2d image coordinates based on angle of view. Plots graph. Many optional parameters for moviemaking ease

    host_ids = hostid(s) of halo to calibrate axes around, as well as color differently
    declination = camera angle measured down from z axis
    azimuthal = camera angle measured counterclockwise from x axis in xy plane
    
    Below are largely movie features 
    
    plot = Whether to display plot on Jupyter or not.
    save = Whether to save plot to file system
    axis_of_rotation = x,y, or z. Axis camera is rotation around.
    angle = Angle from current camera position to default camera position (depends on what type of movie you're making
    limits = 1x4 array of [min image(x), max image(x), min image(y), max image(y)]
    n = index of what image. Stored like 000 for first image, and 010 for 11th image for example.
    
    '''
    #Transforms data into new axes
    new_pos_array_all = np.matmul(transform_matrix, ananke_data_centered.T)

    #Spherical Coordinates
    theta = declination
    phi = azimuthal 
    declination_readable = round(declination*180/np.pi,1)
    azimuthal_readable = round(np.mod(azimuthal*180/np.pi, 360),1)

    #Uses generalized matrix to find 2d projected image from any given camera angle 
    project_matrix = np.array([[-np.sin(phi),                np.cos(phi),             0            ],
                             [-np.cos(theta)*np.cos(phi), -np.cos(theta)*np.sin(phi), np.sin(theta)]])

    project_pos_array_all = np.matmul(project_matrix, new_pos_array_all) #applies 2x3 transformation to 3xn data. Result is 2xn data 

    if plot == True:
        fig,ax  = plt.subplots()
        ax.scatter (project_pos_array_all[0], project_pos_array_all[1], s=1)
        ax.set_title(f"Declination = {declination_readable} degrees and Azimuthal = {azimuthal_readable} degrees")
        ax.set_title
        plt.show()
       

    return declination_readable, azimuthal_readable, project_pos_array_all



In [35]:
for i in range(len(declination_array)):
    d, a, altered_positions=  angle_view(transform_matrix, declination_array[i], azimuthal_array[i], plot=False)
    dwarfcat['px'] = altered_positions.T[:,0]
    dwarfcat['py'] = altered_positions.T[:,1]

    file_path = f"/home/otteleno/MAP/matched_filter/mf_data/rotated_catalogs/{box}_{num}/{box}_{num}_d={d}_a={a}.h5"
    if os.path.exists(file_path):
        os.remove(file_path)

    vframe_new = vaex.from_pandas(dwarfcat)
    vframe_new.export_hdf5(file_path, progress=True)
    






export(hdf5) [########################################] 100.00% elapsed time  :     7.87s =  0.1m =  0.0h
export(hdf5) [########################################] 100.00% elapsed time  :     7.59s =  0.1m =  0.0h
export(hdf5) [########################################] 100.00% elapsed time  :     9.15s =  0.2m =  0.0h
export(hdf5) [########################################] 100.00% elapsed time  :     8.37s =  0.1m =  0.0h
export(hdf5) [########################################] 100.00% elapsed time  :     8.52s =  0.1m =  0.0h
export(hdf5) [########################################] 100.00% elapsed time  :     8.59s =  0.1m =  0.0h
export(hdf5) [########################################] 100.00% elapsed time  :     8.15s =  0.1m =  0.0h
export(hdf5) [########################################] 100.00% elapsed time  :     7.84s =  0.1m =  0.0h
export(hdf5) [########################################] 100.00% elapsed time  :     7.41s =  0.1m =  0.0h
export(hdf5) [################################

In [ ]:
print(len(ananke_data))
print(len(sp))